## Section 1 - Setup and Data Loading

In [ ]:
from __future__ import annotations

import importlib.util
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import f1_score, precision_score, recall_score

EXPERIMENTS_DIR = Path("__file__").resolve().parent if "__file__" in dir() else Path.cwd()

EXPERIMENT_DIRS = {
    "unused_terms" : EXPERIMENTS_DIR / "unused_terms",
    "unbound_variables" : EXPERIMENTS_DIR / "unbound_variables",
    "coherence_answer_section" : EXPERIMENTS_DIR / "coherence_answer_section",
    "compliance_wrt_vocabulary" : EXPERIMENTS_DIR / "compliance_wrt_vocabulary",
    "syntactic_correctness": EXPERIMENTS_DIR / "syntactic_correctness"
}

def _add_to_path(d: Path) -> None:
    s = str(d)
    if s not in sys.path:
        sys.path.insert(0, s)

def _load_metrics_module(name: str, filepath: Path):
    spec = importlib.util.spec_from_file_location(name, filepath)
    mod = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    return mod

print("Experiments directory:", EXPERIMENTS_DIR)

## Load base CSV (Prolog programs)

In [ ]:
BASE_CSV = EXPERIMENTS_DIR / "results_prolog-computed.csv"
df_programs = pd.read_csv(BASE_CSV)

print(f"Base CSV: {BASE_CSV}")
print(f"Shape: {df_programs.shape}")
print(f"Columns: {list(df_programs.columns)}")
print(f"Unique question_index: {df_programs['question_index'].nunique()}")

# TODO: remove clause_per_clause_reward and call it instead in a few cells.
df_programs[["question_index", "model"]].head(5)

In [ ]:
# ANNOTATIONS_XLS = EXPERIMENTS_DIR / "fixes_partially_annotated.xlsx"
ANNOTATIONS_XLS = EXPERIMENTS_DIR / "ann_df.xlsx"
df_annotations = pd.read_excel(ANNOTATIONS_XLS)

print(f"Annotations file: {ANNOTATIONS_XLS.name}")
print(f"Shape: {df_annotations.shape}")
print(f"Columns: {list(df_annotations.columns)}")
print(f"Unique question_index: {df_annotations['question_index'].nunique()}")
df_annotations.head(5)

In [ ]:
fix_cat_counts = df_annotations["fix_tag"].value_counts()
print("fix_tag distribution:")
print(fix_cat_counts.to_string())

## Section 2 - Metric to fix_category mapping

In [ ]:
METRIC_CATEGORY_MAP: dict[str, dict[str]] = {
    'syntactic_correctness': {
        'invalid atom syntax',
        'invalid numeric syntax',
        'general syntax error',
        'invalid syntax annotations',
        'atom/variable confusion',
        'malformed rule structure'
    },
    'unbound_variables': {
        'variable scoping across goals',
        'singleton variable',
        'anonymous variable misuse',
        'missing output binding',
        'redundant variable unification',
        'missing variable unification',
        'incorrect variable unification',
        'unbound variable in head', 
        'unbound variable in rule',
        'unbound variable in facts'
    },
    'unused_terms': {
        'redundant functor wrapping',
        'duplicate solutions via backtracking',
        'redundant condition',
        'redundant fact',
        'duplicate predicate definition',
        'unused variable binding',
        'unused argument',
        'unused predicate'
    },
    'compliance_wrt_vocabulary': {
        'wrong predicate used',
        'wrong predicate call order',
        'predicate overloading',
        'inconsistent head body unification',
        'malformed predicate/rule structure',
        'missing entry point',
        'missing predicate definition',
        'undefined predicate call',
        'atom/identifier typo',
        'wrong predicate name',
        'argument value mismatch',
        'argument structure/type mismatch',
        'argument order mismatch',
        'arity mismatch'
    }
}

for metric, cats in METRIC_CATEGORY_MAP.items():
    print(f"  {metric:35s} -> {cats}")

In [ ]:
all_tags = []

for x in METRIC_CATEGORY_MAP.values():
    for y in x:
        all_tags.append(y)

df_val = df_annotations['fix_tag'].value_counts().reset_index()
df_val = df_val[df_val['fix_tag'].isin(all_tags)]
df_val


In [ ]:
import matplotlib.pyplot as plt

x_col = df_val.columns[0] # Usually the categories ('index' or 'fix_tag')
y_col = df_val.columns[1] # Usually the counts ('fix_tag' or 'count')

# Create the plot
plt.figure(figsize=(10, 6)) # Adjust width and height as needed
plt.bar(df_val[x_col], df_val[y_col], color='steelblue')

# Add labels and title
plt.title('Frequency of Fix Tags')
plt.xlabel('Fix Tag')
plt.ylabel('Count')

# Rotate x-axis labels so they are readable and don't overlap
plt.xticks(rotation=45, ha='right')

# Adjust layout to prevent labels from being cut off
plt.tight_layout()

# Display the plot
plt.show()

## Section 3 - Build ground-truth labels

For each `question_index` and each metric, build a binary label:
* `y_true = 1` if any of that question's `fix_category` entries are in the metric's set
* `y_true = 0` otherwise

In [ ]:
qi_to_categories: dict[int, set[str]] = (
    df_annotations
        .groupby("question_index")['fix_tag']
        .apply(lambda s: set(s.dropna().tolist()))
        .to_dict()
)

sample_qi = list(qi_to_categories.keys())[:3]
for qi in sample_qi:
    print(f"question_index={qi}: {qi_to_categories[qi]}")

In [ ]:
unique_qis = sorted(df_programs['question_index'].unique())

ground_truth: dict[str, list[int]] = {"question_index": unique_qis}

for metric, target_cats in METRIC_CATEGORY_MAP.items():
    labels = []
    for qi in unique_qis:
        question_cats = qi_to_categories.get(qi, set())
        print(question_cats)
        print(target_cats)
        labels.append(1 if question_cats & target_cats else 0)

    ground_truth[f"y_true_{metric}"] = labels

df_gt = pd.DataFrame(ground_truth)

print("Ground-truth label distribution (# positives out of 100):")
for metric in METRIC_CATEGORY_MAP:
    col = f"y_true_{metric}"
    n_pos = df_gt[col].sum()
    print(f"  {metric:35s}  positives={n_pos:3d}  negatives={100-n_pos:3d}")

df_gt.head()

## Section 4 - Compute Metrics

Run all 6 analyzers on the 100 Prolog programs.

Note: `coherence` requires an LLM API key ; `compliance_wrt_ontology` requires loading a SentenceTransformer model. Set `SKIP_COHERENCE` / `SKIP_COMPLIANCE` to `True` to skip them.

A metric **flags** a program when `score < treshold` - a lower score signals a detected issue.

In [ ]:

EMBEDDING_MODEL_NAME = "all-MiniLM-L6-V2"
ONTOLOGY_PATH = EXPERIMENT_DIRS['compliance_wrt_vocabulary'] / "vocabulary.json"

In [ ]:
EXPERIMENT_DIRS

In [ ]:
# syntactic correctness TODO:
_add_to_path(EXPERIMENT_DIRS["syntactic_correctness"])
from syntactic_correctness.syntactic_correctness_analyzer import SyntacticCorrectnessAnalyzer
_syn_mod = _load_metrics_module("syn_metrics", EXPERIMENT_DIRS['syntactic_correctness'] / "metrics.py")
syn_score_fn = _syn_mod.syntactic_correctness_score

# predicate utilization
_add_to_path(EXPERIMENT_DIRS["unused_terms"])
from unused_terms.unused_terms_analyzer import UnusedTermsAnalyzer  # noqa
_ut_mod  = _load_metrics_module("ut_metrics", EXPERIMENT_DIRS["unused_terms"] / "metrics.py")
ut_score_fn = _ut_mod.predicate_utilization_score

# clause cleanliness
_add_to_path(EXPERIMENT_DIRS["unbound_variables"])
from unbound_variables.unbound_variables_analyzer import UnboundVariablesAnalyzer  # noqa
_uv_mod  = _load_metrics_module("uv_metrics", EXPERIMENT_DIRS["unbound_variables"] / "metrics.py")
uv_score_fn = _uv_mod.clause_cleanliness_score

# compliance wrt vocabulary
_add_to_path(EXPERIMENT_DIRS['compliance_wrt_vocabulary'])
from compliance_wrt_vocabulary.type_compliance_analyzer import TypeComplianceAnalyzer, load_ontology
ontology = load_ontology(ONTOLOGY_PATH)
_tc_mod = _load_metrics_module("tc_metrics", EXPERIMENT_DIRS['compliance_wrt_vocabulary'] / "metrics.py")
tc_score_fn = _tc_mod.type_compliance_score

print("Core analyzers imported successfully.")

In [ ]:
# quick function to get the appropriate syntax printed for the paper
def get_latex_string(strings):
    return ",".join([f"\\textit{{{s}}} " for s in strings])

for metric, strings in METRIC_CATEGORY_MAP.items():
    print("---", metric)
    print(get_latex_string(strings))

In [ ]:
from tqdm.auto import tqdm
from transformers import logging
from sentence_transformers import SentenceTransformer

logging.set_verbosity_error()
logging.disable_progress_bar()

# Pre-load the embedding model once for compliance_wrt_ontology
# (avoids reloading the SentenceTransformer on every iteration)
_st_model = SentenceTransformer('all-MiniLM-L6-v2')
_tc_type_embeddings = None  # will be cached after first TypeComplianceAnalyzer call

records = []

for _, row in tqdm(df_programs.iterrows(), total=len(df_programs), desc="computing metrics"):
    qi = row['question_index']
    code = str(row['reasoning']) if pd.notna(row['reasoning']) else ""
    answer = str(row['answer']) if pd.notna(row['answer']) else ""

    # syntactic correctness
    # FIXME: for the moment, we re-use the existing column, but we'll want to compute it on the fly
    # syn = float(row['clause_per_clause_reward']) if pd.notna(row['clause_per_clause_reward']) else float("nan")
    syn = syn_score_fn(SyntacticCorrectnessAnalyzer(code).analyze()) if code.strip() else 0.0

    # predicate utilization
    ut = ut_score_fn(UnusedTermsAnalyzer(code).analyze()) if code.strip() else 0.0

    # clause cleanliness
    uv = uv_score_fn(UnboundVariablesAnalyzer(code).analyze()) if code.strip() else 0.0

    # compliance wrt vocabulary
    if code.strip():
        try:
            tc_analyzer = TypeComplianceAnalyzer(
                code=code, ontology=ontology,
                model=_st_model, type_embeddings=_tc_type_embeddings,
            )
            tc_result = tc_analyzer.analyze()
            # Cache type embeddings after first call
            if _tc_type_embeddings is None:
                _tc_type_embeddings = tc_analyzer._type_embeddings
            tc = tc_score_fn(tc_result)
            print("tc", tc)
            tc = float(tc) if tc is not None else float("nan")
        except Exception:
            tc = float("nan")
    else:
        tc = float("nan")

    records.append({
        "question_index": qi,
        "syntactic_correctness": syn,
        "unused_terms": ut,
        "unbound_variables": uv,
        "compliance_wrt_vocabulary": tc
    })

df_scores = pd.DataFrame(records)
print("\nComputed scores (first 5 rows):")
df_scores.head()

In [ ]:
METRIC_CATEGORY_MAP

In [ ]:
metric_cols = list(METRIC_CATEGORY_MAP.keys())
print("Score statistics:")
df_scores[metric_cols].describe().round(4)

In [ ]:
# merge scores with ground truth for downstream analysis
df_analysis = df_gt.merge(df_scores, on='question_index', how="inner")
print(f"Analysis DataFrame shape: {df_analysis.shape}")
df_analysis.head(3)

## Section 5 - Multi-Threshold Classification Analysis

For each metric, we sweep the flag threshold from 0 to 1:
* `y_pred = 1` (metric flags an issue) when `score < threshold`
* `y_pred = 0` otherwise

We plot F1, precision, and recall as a function of the threshold and highlight the threshold that maximizes F1.

In [ ]:
THRESHOLDS = np.linspace(0.0, 1.0, 51) 

def classification_at_threshold(y_true, y_score, threshold: float) -> dict:
    
    y_pred = (y_score < threshold).astype(int)

    print(f"y_score = {y_score} ; threshold = {threshold} ; y_pred = {y_pred}")

    zw = dict(zero_division=0)
    return {
        "threshold": threshold,
        "f1": f1_score(y_true, y_pred, **zw),
        "precision": precision_score(y_true, y_pred, **zw),
        "recall": recall_score(y_true, y_pred, **zw),
        "tp": int(((y_pred == 1) & (y_true == 1)).sum()),
        "fp": int(((y_pred == 1) & (y_true == 0)).sum()),
        "tn": int(((y_pred == 0) & (y_true == 0)).sum()),
        "fn": int(((y_pred == 0) & (y_true == 1)).sum())
    }

curves: dict[str, pd.DataFrame] = {}

for metric in metric_cols:

    col_true = f"y_true_{metric}"
    col_score = metric

    print("metric", metric)

    # drop rows where score is nan
    mask = df_analysis[col_score].notna()
    y_true = df_analysis.loc[mask, col_true].values
    y_score = df_analysis.loc[mask, col_score].values

    rows = [classification_at_threshold(y_true, y_score, t) for t in THRESHOLDS]
    curves[metric] = pd.DataFrame(rows)

print("Threshold sweep complete for all metrics.")

In [ ]:
METRIC_CATEGORY_MAP

In [ ]:
import matplotlib.pyplot as plt
plt.rcParams["font.family"] = "sans-serif"

METRIC_LABELS = {
    "syntactic_correctness": "Syntactic Correctness",
    "unused_terms": "Unused Terms",
    "unbound_variables": "Unbound Variables",
    "compliance_wrt_vocabulary": "Compliance w.r.t. Vocabulary",
}

best_thresholds: dict[str, float] = {}

# 1. Create a 2x3 grid of subplots
fig, axes = plt.subplots(nrows=2, ncols=3, figsize=(15, 9))
axes = axes.flatten() # Flatten for easy iteration

# 2. Loop through metrics to populate the subplots
for i, metric in enumerate(metric_cols):
    ax = axes[i]
    curve = curves[metric]
    
    # Identify the best F1 score and its corresponding threshold
    best_idx = curve['f1'].idxmax()
    best_t = curve.loc[best_idx, "threshold"]
    best_f1 = curve.loc[best_idx, 'f1']
    best_thresholds[metric] = best_t

    # Trace: Precision
    ax.plot(curve['threshold'], curve['precision'], 
            color='steelblue', lw=2, label='Precision')

    # Trace: Recall
    ax.plot(curve['threshold'], curve['recall'], 
            color='coral', lw=2, label='Recall')

    # Trace: F1 Score (slightly thicker)
    ax.plot(curve['threshold'], curve['f1'], 
            color='seagreen', lw=3, label='F1')

    # Vertical line for the best threshold
    ax.axvline(x=best_t, color='gray', linestyle='--', lw=1.5, alpha=0.7)

    # Visual annotation for the best F1
    ax.annotate(
        f"Best F1={best_f1:.3f}\n@ t={best_t:.2f}",
        xy=(best_t, best_f1),
        xytext=(30, 25), textcoords='offset points', # Offset the text like Plotly's ax/ay
        arrowprops=dict(arrowstyle="->", color="seagreen", lw=1.5),
        color="seagreen", fontsize=11, fontweight='bold'
    )
    
    # Academic formatting for specific axes
    title = METRIC_LABELS.get(metric, metric)
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.set_xlabel("Threshold (flag if score < t)", fontsize=11)
    ax.set_ylabel("Score", fontsize=11)
    
    # Fix axes limits
    ax.set_ylim([-0.05, 1.05])
    
    # Clean up the spines (removes top and right borders for a cleaner look)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

# 3. Global Design & Cleanup

# Hide any unused subplots (e.g., the 6th plot if you only have 5 metrics)
for j in range(len(metric_cols), len(axes)):
    axes[j].axis('off')

# Extract legend handles from the first axis to create a single global legend
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, 
           loc='upper center', 
           ncol=3, 
           bbox_to_anchor=(0.5, 1.05), # Positioned above the plots
           fontsize=10, 
           frameon=False)

# Adjust layout to prevent overlap, leaving room at the top for the legend
plt.tight_layout(rect=[0, 0, 1, 0.95])

# Save and show
plt.savefig('pr_f1_curves.png', bbox_inches='tight')
plt.show()

## Section 6 - Summary Table

In [ ]:
summary_rows = []

for metric in metric_cols:
    col_true = f"y_true_{metric}"
    col_score = metric
    mask = df_analysis[col_score].notna()
    y_true = df_analysis.loc[mask, col_true].values
    y_score = df_analysis.loc[mask, col_score].values

    n_evaluable = int(mask.sum())
    n_positive = int(y_true.sum())
    n_negative = n_evaluable - n_positive
    
    curve = curves[metric]
    best_idx = curve['f1'].idxmax()
    best_row = curve.loc[best_idx]

    summary_rows.append({
        "Metric": METRIC_LABELS[metric],
        "Best threshold": round(best_row['threshold'], 2),
        "Best f1": round(best_row['f1'], 2),
        "Best precision": round(best_row['precision'], 2),
        "Best recall": round(best_row['recall'], 2),
        "TP": int(best_row['tp']),
        "FP": int(best_row['fp']),
        "FN": int(best_row['fn']),
        "TN": int(best_row['tn']),
        "# positives": n_positive,
        "# negatives": n_negative,
        "# evaluable": n_evaluable
    })

df_summary = pd.DataFrame(summary_rows).set_index("Metric")
df_summary

## Section 7 - Error Analysis

In [ ]:
def get_errors_at_best_threshold(metric: str) -> dict:
    """Return FP and FN question indices at the best F1 threshold for *metric*."""
    col_true  = f"y_true_{metric}"
    col_score = metric
    mask   = df_analysis[col_score].notna()
    sub    = df_analysis[mask].copy()

    best_t  = best_thresholds[metric]
    sub["y_pred"] = (sub[col_score] < best_t).astype(int)

    fp_qis = sub.loc[(sub["y_pred"] == 1) & (sub[col_true] == 0), "question_index"].tolist()
    fn_qis = sub.loc[(sub["y_pred"] == 0) & (sub[col_true] == 1), "question_index"].tolist()
    return {"false_positives": fp_qis, "false_negatives": fn_qis}


print("Error analysis at best threshold:")
print(f"{'Metric':<35} {'FP':>4}  {'FN':>4}  {'FP qis'}")
print("-" * 80)
errors_by_metric = {}
for metric in metric_cols:
    errs = get_errors_at_best_threshold(metric)
    errors_by_metric[metric] = errs
    print(f"{METRIC_LABELS[metric]:<35} {len(errs['false_positives']):>4}  "
          f"{len(errs['false_negatives']):>4}  "
          f"fp={errs['false_positives']}")

In [ ]:
def show_frequent_tags_qis(qis):
    dfqis = df_annotations[df_annotations['question_index'].isin(qis)]
    
    return dfqis['fix_tag'].value_counts()

metric_to_show = "unused_terms"
print(f"--- METRIC = {metric_to_show}")
print("tags assigned to the metric: ", METRIC_CATEGORY_MAP[metric_to_show])

errs = get_errors_at_best_threshold(metric_to_show)
frequent_tags_false_positives = show_frequent_tags_qis(errs['false_positives'])
print("frequent_tags-FP", frequent_tags_false_positives)

frequent_tags_false_negatives = show_frequent_tags_qis(errs['false_negatives'])
print("frequent_tags-FN", frequent_tags_false_negatives)



In [ ]:
def inspect_question(question_index: int, metric: str | None = None) -> None:
    """Print the Prolog code and annotations for a given question_index."""
    rows = df_programs[df_programs["question_index"] == question_index]
    if rows.empty:
        print(f"question_index={question_index} not found.")
        return

    code = rows.iloc[0]["reasoning"]
    cats = qi_to_categories.get(question_index, set())

    print(f"=== question_index = {question_index} ===")
    print(f"fix_categories : {cats}")

    if metric is not None:
        score  = df_scores[df_scores["question_index"] == question_index][metric].values
        best_t = 0.5 #best_thresholds[metric]
        y_true = df_gt[df_gt["question_index"] == question_index][f"y_true_{metric}"].values
        flagged = score[0] < best_t if len(score) else "N/A"
        print(f"metric         : {metric}")
        print(f"score          : {score[0] if len(score) else 'N/A':.4f}")
        print(f"threshold      : {best_t:.2f}")
        print(f"flagged        : {flagged}")
        print(f"y_true         : {int(y_true[0]) if len(y_true) else 'N/A'}")

    print("\n--- Prolog code ---")
    print(code)
    print()

In [ ]:
# printing the false negatives for a metric score, understanding the problem happening with the metric
metric_to_show = "unused_terms"

errs = get_errors_at_best_threshold(metric_to_show)
print(errs['false_negatives'])

for qi in errs['false_negatives']:
    inspect_question(qi, metric=metric_to_show)

In [ ]:
metric_to_show = "unused_terms"

fn_ir = errors_by_metric[metric_to_show]["false_negatives"]
print(f"False negatives for {metric_to_show}: {fn_ir}")
for qi in fn_ir[:3]:   # show up to 3
    inspect_question(qi, metric=metric_to_show)

In [ ]:
metric_to_show = "unused_terms"

fp_ir = errors_by_metric[metric_to_show]["false_positives"]
print(f"False positives for {metric_to_show}: {fp_ir}")
for qi in fp_ir[:3]:
    inspect_question(qi, metric=metric_to_show)

---
## Section 8 — ROC Curves

One ROC curve per metric, computed with `sklearn.metrics.roc_curve`.

**Convention**: lower metric scores indicate a detected issue (positive class),
so we pass `1 - score` to `roc_curve` to get the standard orientation.

In [ ]:
# same plot in matplotlib with basic, academic style
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc

METRIC_LABELS = {
    "syntactic_correctness": "Syntactic Correctness",
    "unused_terms": "Unused Terms",
    "unbound_variables": "Unbound Variables",
    "compliance_wrt_ontology": "Compliance w.r.t. Vocabulary",
}

# Create a 2x3 grid of subplots
fig, axes = plt.subplots(nrows=2, ncols=3, figsize=(15, 10))
# Flatten the 2D array of axes for easy iteration
axes = axes.flatten()

for i, metric in enumerate(metric_cols):
    ax = axes[i]
    col_true = f"y_true_{metric}"
    col_score = metric

    # Drop rows where score is NaN
    mask = df_analysis[col_score].notna()
    y_true = df_analysis.loc[mask, col_true].values
    y_score = df_analysis.loc[mask, col_score].values

    # Lower score => positive class, so flip scores for sklearn
    fpr, tpr, thresholds = roc_curve(y_true, 1 - y_score)
    roc_auc = auc(fpr, tpr)

    # Best threshold = maximises Youden's J (TPR - FPR)
    j_scores = tpr - fpr
    best_idx = j_scores.argmax()
    best_fpr, best_tpr = fpr[best_idx], tpr[best_idx]
    
    # Convert back to the original metric scale
    best_threshold_original = 1 - thresholds[best_idx]

    label = METRIC_LABELS.get(metric, metric)

    # 1. Plot ROC curve
    ax.plot(fpr, tpr, color='steelblue', lw=2, label=f'AUC = {roc_auc:.3f}')

    # 2. Plot Diagonal (random classifier)
    ax.plot([0, 1], [0, 1], color='gray', lw=1, linestyle='--')

    # 3. Plot Best threshold marker
    ax.plot(best_fpr, best_tpr, marker='x', color='red', markersize=8, 
            label=f'Best Threshold')
    
    # 4. Annotate the threshold value near the marker
    ax.annotate(f't={best_threshold_original:.3f}', 
                xy=(best_fpr, best_tpr), 
                xytext=(8, -12), textcoords='offset points', 
                color='red', fontsize=10)

    # 5. Academic formatting
    ax.set_title(label, fontsize=12, fontweight='bold')
    ax.set_xlabel('False Positive Rate', fontsize=11)
    ax.set_ylabel('True Positive Rate', fontsize=11)
    
    # Force axes limits and square aspect ratio
    ax.set_xlim([-0.02, 1.02])
    ax.set_ylim([-0.02, 1.02])
    ax.set_aspect('equal', adjustable='box')
    
    # Add legend to the lower right
    ax.legend(loc='lower right', fontsize=10, frameon=True)

    print(f"{label}:  AUC = {roc_auc:.4f}  |  "
          f"Best threshold = {best_threshold_original:.4f}  |  "
          f"TPR = {best_tpr:.3f}  |  FPR = {best_fpr:.3f}")

# Hide any unused subplots (e.g., the 6th plot if you only have 5 metrics)
for j in range(len(metric_cols), len(axes)):
    axes[j].axis('off')

# Adjust layout so labels don't overlap
plt.tight_layout()

# Save as a high-resolution vector graphic for your paper (optional)
# plt.savefig('roc_curves.pdf', format='pdf', bbox_inches='tight')

# Display the plot
plt.savefig('roc_curves.png')
plt.show()

In [ ]:
metric_to_show = "unused_terms"
print(f"--- METRIC = {metric_to_show}")
print("tags assigned to the metric: ", METRIC_CATEGORY_MAP[metric_to_show])

errs = get_errors_at_best_threshold(metric_to_show)
frequent_tags_false_positives = show_frequent_tags_qis(errs['false_positives'])
print("frequent_tags-FP", frequent_tags_false_positives)

frequent_tags_false_negatives = show_frequent_tags_qis(errs['false_negatives'])
print("frequent_tags-FN", frequent_tags_false_negatives)

In [ ]:
metric_cols

In [ ]:
metric_to_show = "syntactic_correctness"

errs = get_errors_at_best_threshold(metric_to_show)

print(f"metric is: {metric_to_show}")

classical_issues_metric = METRIC_CATEGORY_MAP[metric_to_show]
print(classical_issues_metric)


# question index whose Prolog are classified as FP
print(errs['false_positives'])
for fp_qi in errs['false_positives']:
    prolog = df_programs[df_programs['question_index'] == fp_qi].iloc[0]['reasoning']
    tags = df_annotations[df_annotations['question_index'] == fp_qi]['fix_tag'].to_list()
    print("---------------")
    print(f"The following program (qi={fp_qi}) has been classified as FP (metric={metric_to_show}).")
    print("(meaning it should have one of the tags associated to the metric)")
    print(prolog)

# question index whose Prolog are classified as FN
print(errs['false_negatives'])
for fn_qi in errs['false_negatives']:
    prolog = df_programs[df_programs['question_index'] == fn_qi].iloc[0]['reasoning']
    tags = df_annotations[df_annotations['question_index'] == fn_qi]['fix_tag'].to_list()
    print("---------------")
    print(f"The following program (qi={fn_qi}) has been classified as FN (metric={metric_to_show}).")
    print(f"tags that should not have been included to not be classified as FN according to the metric: {set(tags) & set(classical_issues_metric)}")
    print(prolog)

In [ ]:
df_programs[df_programs['question_index'] == 63]